## Invesigating effects of convolving CHIME Q,U maps to DRAGONS beam
### Mar 12
### Next step: new CHIME FD cubes at both CHIME and DRAGONS resolution

In [ ]:
import numpy as np
from astropy.io import fits as fits
import healpy as hp
from astropy.wcs import WCS
from reproject import reproject_from_healpix, reproject_to_healpix
import matplotlib.pyplot as plt

In [ ]:
def convolve_to_beam(map_in_hpx, beam_small, beam_big):

    fwhm = np.sqrt(beam_big**2 - beam_small**2)

    map_in_hpx[np.isnan(map_in_hpx)] = hp.UNSEEN
    
    map_out_hpx = hp.sphtfunc.smoothing(map_in_hpx, fwhm=np.radians(fwhm))

    return map_out_hpx


## Read in CHIME and DRAGONS data
### (DRAGONS only for comparison)

In [ ]:
chime_hduQ = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/Q_400_729_Mar2024_new_gal_fix.fits')
chime_Q = chime_hduQ[0].data

chime_hduU = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/U_400_729_Mar2024_new_gal_fix.fits')
chime_U = chime_hduU[0].data

chime_hdr  = chime_hduQ[0].header
freq_chime = WCS(chime_hdr).all_pix2world(0,0,range(chime_Q.shape[0]),0)[2]
chime_hdr_2D = WCS(chime_hdr).dropaxis(2).to_header()

dragons_hduQ = fits.open('/srv/data/dragons/dragons-canfar-Nov2025/frequency_cubes/dragons_freqK_gal_conv_q.hpx.fits')
dragons_Q_hpx = dragons_hduQ[0].data

dragons_hduU = fits.open('/srv/data/dragons/dragons-canfar-Nov2025/frequency_cubes/dragons_freqK_gal_conv_u.hpx.fits')
dragons_U_hpx = dragons_hduU[0].data

freq_dragons = WCS(dragons_hduQ[0].header).all_pix2world(0,range(dragons_Q_hpx.shape[0]),0)[1]

## Make CHIME beam profile

In [ ]:
lbd_arr = (3e8)/freq_chime

x_beam = (0.85*lbd_arr/66)*(180/np.pi)*60
y_beam = (0.85*lbd_arr/78)*(180/np.pi)*60

fig,ax = plt.subplots(1,figsize=(8,3))

ax.plot(freq_chime/1e6,x_beam,label='East-West',linewidth=2)
ax.plot(freq_chime/1e6,y_beam,label='North-South',linewidth=2)
ax.plot(freq_chime/1e6,(y_beam+x_beam)/2,label='mean',linewidth=2)
ax.set_ylim(10,35)
ax.set_xlim(400,800)
ax.grid()
ax.set_xlabel('Frequency (MHz)',fontsize=10)
ax.set_ylabel('Beam FWHM (arcmin)',fontsize=10)
ax.legend(fontsize=10)
ax.tick_params(axis='both', labelsize=10)

## Run comparison at single frequency

In [ ]:
def compare_maps(fpick, stokes):

    nside = 512
    idx_chime = abs(fpick-freq_chime).argmin()
    idx_dragons = abs(fpick-freq_dragons).argmin()
    print(freq_chime[idx_chime])
    print(freq_dragons[idx_dragons])

    if stokes == 'Q':
        chime_hpx, footprint = reproject_to_healpix((chime_Q[idx_chime], chime_hdr_2D), coord_system_out='G',nside=nside)
        chime_hpx_smooth = convolve_to_beam(chime_hpx, 0.333, 3.6)
        dragons_hpx = dragons_Q_hpx[idx_dragons]
    if stokes == 'U':
        chime_hpx, footprint = reproject_to_healpix((chime_U[idx_chime], chime_hdr_2D), coord_system_out='G',nside=nside)
        chime_hpx_smooth = convolve_to_beam(chime_hpx, 0.333, 3.6)
        dragons_hpx = dragons_U_hpx[idx_dragons]

    chime_hpx_smooth = hp.pixelfunc.ud_grade(chime_hpx_smooth, 64)
    chime_hpx = hp.pixelfunc.ud_grade(chime_hpx, 64)
    dragons_hpx = hp.pixelfunc.ud_grade(dragons_hpx, 64)

    #hp.mollview(chime_hpx,        min=-2, max=2, cmap='RdBu_r')
    #hp.mollview(chime_hpx_smooth, min=-2, max=2, cmap='RdBu_r')
    #hp.mollview(dragons_hpx,      min=-2, max=2, cmap='RdBu_r')

    print(chime_hpx_smooth.shape)
    print(dragons_hpx.shape)

    return chime_hpx, chime_hpx_smooth, dragons_hpx

In [ ]:
chime_hpx, chime_hpx_smooth, dragons_hpx = compare_maps(610e6, 'U')

fig, ax = plt.subplots(1,2,figsize=(11,5))
ax[0].scatter(chime_hpx, dragons_hpx, s=0.1)
ax[1].scatter(chime_hpx_smooth, dragons_hpx, s=0.1)
for i in range(0,2):
    ax[i].set_xlim(-1.5,1.5)
    ax[i].set_ylim(-1.5,1.5)
    ax[i].set_aspect('equal')
    ax[i].grid(True)

## Make smoothed CHIME cube

In [ ]:
def convolve_all_chime(chime_Q, chime_U, chime_beam, nside):

    chime_Q_hpx_conv = np.empty([chime_Q.shape[0],hp.nside2npix(nside)])
    chime_U_hpx_conv = np.empty([chime_U.shape[0],hp.nside2npix(nside)])
    
    for i in range(0,chime_Q.shape[0]):
    #for i in range(0,3):

        print(i,chime_beam[i])
        
        chime_hpxQ, footprint = reproject_to_healpix((chime_Q[i], chime_hdr_2D), coord_system_out='G',nside=nside)
        chime_hpxU, footprint = reproject_to_healpix((chime_U[i], chime_hdr_2D), coord_system_out='G',nside=nside)

        chime_Q_hpx_conv[i] = convolve_to_beam(chime_hpxQ, chime_beam[i], 3.6)
        chime_U_hpx_conv[i] = convolve_to_beam(chime_hpxU, chime_beam[i], 3.6)

    return chime_Q_hpx_conv, chime_U_hpx_conv
    

In [ ]:
def chime_hpx_hdr(hdr_hpx_in,hdr_car_chime,nside):

    hpx_hdr = hdr_hpx_in.copy()
    hpx_hdr['NAXIS1'] = hp.nside2npix(nside)
    hpx_hdr['NAXIS2'] = hdr_car_chime['NAXIS3']
    hpx_hdr['NSIDE']  = nside
    hpx_hdr['CRPIX2'] = hdr_car_chime['CRPIX3']
    hpx_hdr['CDELT2'] = hdr_car_chime['CDELT3']
    hpx_hdr['CRVAL2'] = hdr_car_chime['CRVAL3']
    hpx_hdr['BUNIT']  = 'Jy/beam'
    del hpx_hdr['DATE']
    del hpx_hdr['INSTRUME']
    del hpx_hdr['COMMENT']
    
    return hpx_hdr

In [ ]:
nside = 512

In [ ]:
hpx_hdr_chime = chime_hpx_hdr(dragons_hduU[0].header,chime_hdr,nside)

chime_beam_deg = ((y_beam+x_beam)/2)/60.
chime_Q_hpx_conv, chime_U_hpx_conv = convolve_all_chime(chime_Q, chime_U, chime_beam_deg, nside)

fits.writeto('/srv/data/chime/chime_IQUV_Mar2024_400_729/Q_400_729_Mar2026_gal_conv.fits', 
                 chime_Q_hpx_conv, hpx_hdr_chime, overwrite=False)
fits.writeto('/srv/data/chime/chime_IQUV_Mar2024_400_729/U_400_729_Mar2026_gal_conv.fits', 
                 chime_U_hpx_conv, hpx_hdr_chime, overwrite=False)


In [ ]:
hp.mollview(chime_Q_hpx_conv[1],min=-1,max=1)
hp.mollview(chime_U_hpx_conv[1],min=-1,max=1)